# Causal Fairness — Dev Log

## Objetivo

`fairness_audit` (V2) mede correlação bruta; este módulo estratifica por
uma variável confundidora e detecta o **Paradoxo de Simpson** — quando o
veredito agregado diverge do veredito dentro de cada estrato.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.causal_fairness.stratified import stratified_fairness_audit

# Exemplo clássico estilo Berkeley: M concentra no depto fácil, F no difícil.
records = (
    [{"gender": "M", "dept": "A", "admitted": i < 80} for i in range(90)]
    + [{"gender": "F", "dept": "A", "admitted": i < 9} for i in range(10)]
    + [{"gender": "M", "dept": "B", "admitted": i < 1} for i in range(10)]
    + [{"gender": "F", "dept": "B", "admitted": i < 10} for i in range(90)]
)
result = stratified_fairness_audit(records, "admitted", "gender", "dept")
print(f"Agregado: justo={result.aggregate.overall_fair} | taxas={result.aggregate.selection_rates}")
for s in result.strata:
    print(f"  Estrato '{s.stratum}' (n={s.sample_size}): justo={s.fairness.overall_fair} | taxas={s.fairness.selection_rates}")
print()
print(f"Paradoxo de Simpson detectado? {result.simpsons_paradox_detected}")
print(result.summary)

Agregado: justo=False | taxas={'M': 0.81, 'F': 0.19}
  Estrato 'A' (n=100): justo=True | taxas={'M': 0.8889, 'F': 0.9}
  Estrato 'B' (n=100): justo=True | taxas={'M': 0.1, 'F': 0.1111}

Paradoxo de Simpson detectado? True
Paradoxo de Simpson DETECTADO: o veredito agregado (injusto) diverge do veredito em pelo menos um estrato de 'dept' — a disparidade observada pode ser causada (ou mascarada) por essa variável confundidora, não pelo atributo protegido em si.


O exemplo clássico funciona exatamente como a literatura prevê: o
agregado mostra forte disparidade (81% vs 19%), mas em CADA departamento
individualmente a taxa de F é igual ou maior que a de M. A disparidade
agregada é inteiramente explicada pela distribuição desigual de candidatos
entre departamentos (o confundidor), não por discriminação dentro de cada
departamento.

## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/causal_fairness/tests -v
```

6/6 testes passando.